In [1]:
# Instala condacolab, que permite tener un entorno conda/mamba real en Colab
!pip install -q condacolab
import condacolab
condacolab.install()   # Esto reinicia el kernel de Colab automáticamente
#asegurate de volver a correr hasta que apareza Everything looks OK!

✨🍰✨ Everything looks OK!


In [2]:
# Revisa qué versión de Python quedó "fijada" (pin) por condacolab
!cat /usr/local/conda-meta/pinned

python 3.12.*
python_abi 3.12.* *cp312*
cuda-version 12.*


In [3]:
# Elimina el pin conflictivo: así mamba puede resolver libremente
# las dependencias de sage sin chocar contra una versión de Python fija
!rm -f /usr/local/conda-meta/pinned

In [4]:
# Instala SageMath completo vía mamba (tarda varios minutos, es normal)
!mamba install -y -c conda-forge sage


Looking for: ['sage']

[+] 0.0s
conda-forge/linux-64  ⣾  
conda-forge/noarch    ⣾  [+] 0.1s
conda-forge/linux-64  ⣾  
conda-forge/noarch     1%[+] 0.2s
conda-forge/linux-64   5%
conda-forge/noarch    14%[+] 0.3s
conda-forge/linux-64  12%
conda-forge/noarch    28%[+] 0.4s
conda-forge/linux-64  18%
conda-forge/noarch    39%[+] 0.5s
conda-forge/linux-64  22%
conda-forge/noarch    47%[+] 0.6s
conda-forge/linux-64  27%
conda-forge/noarch    57%[+] 0.7s
conda-forge/linux-64  32%
conda-forge/noarch    62%[+] 0.8s
conda-forge/linux-64  34%
conda-forge/noarch    72%[+] 0.9s
conda-forge/linux-64  39%
conda-forge/noarch    82%[+] 1.0s
conda-forge/linux-64  44%
conda-forge/noarch    92%conda-forge/noarch                                
[+] 1.1s
conda-forge/linux-64  49%[+] 1.2s
conda-forge/linux-64  58%[+] 1.3s
conda-forge/linux-64  66%[+] 1.4s
conda-forge/linux-64  73%[+] 1.5s
conda-forge/linux-64  80%[+] 1.6s
conda-forge/linux-64  83%[+] 1.7s
conda-forge/linux-64  90%[+] 1.8s
conda-forge/linux-

In [5]:
import subprocess, tempfile, os, json

def ejecutar_sage(codigo_sage: str, timeout: int = 1800) -> str:
    """
    Ejecuta código Sage como proceso externo (binario 'sage'), evitando
    el import directo dentro del kernel de Colab (incompatible por
    versión de Python). Devuelve la salida estándar como texto.
    """
    with tempfile.NamedTemporaryFile(mode='w', suffix='.sage', delete=False) as f:
        f.write(codigo_sage)
        ruta = f.name

    try:
        resultado = subprocess.run(
            ["sage", ruta], capture_output=True, text=True, timeout=timeout
        )
    finally:
        os.unlink(ruta)

    if resultado.returncode != 0:
        raise RuntimeError(f"Error ejecutando Sage:\n{resultado.stderr}")

    return resultado.stdout

In [6]:
_PLANTILLA_SAGE_BATCH = r'''
from sage.all import *
import json

var_name = "%%VAR%%"
polinomios = json.loads(r"""%%POLYS_JSON%%""")
R = PolynomialRing(QQ, var_name)

resultados = []
for poly_str in polinomios:
    entry = {"polinomio": poly_str}
    try:
        f = R(poly_str)
        if not f.is_irreducible():
            raise ValueError("El polinomio no es irreducible.")

        n = f.degree()
        K = NumberField(f, 'a')
        disc = K.discriminant()
        sig = K.signature()
        ramified = sorted(disc.prime_factors())

        G = f.galois_group()
        grupo_perm = G.group() if hasattr(G, "group") else G
        Gfresh = PermutationGroup(grupo_perm.gens(), canonicalize=False)

        grado_perm = Gfresh.degree()
        orden_galois = Gfresh.order()
        # libgap (enlace C directo) evita el bug de parsing de la consola de texto 'gap.xxx()'
        t_num = Integer(libgap.TransitiveIdentification(Gfresh.gap()))
        label = "%sT%s" % (grado_perm, t_num)

        try:
            nombre = str(Gfresh.structure_description())
        except Exception:
            nombre = label

        es_abeliano = bool(Gfresh.is_abelian())
        es_ciclico = bool(Gfresh.is_cyclic())
        es_galois = bool(K.is_galois())

        # float(...) exterior obligatorio: RealNumber de Sage no es serializable a JSON
        root_disc = float(float(abs(disc)) ** (1.0 / n))

        if es_galois:
            galois_root_disc = root_disc
            galois_root_disc_exacto = None
            if len(ramified) == 1:
                p = ramified[0]
                e = disc.valuation(p)
                galois_root_disc_exacto = "%s^(%s/%s)" % (p, e, n)
        else:
            galois_root_disc = None
            galois_root_disc_exacto = None
            if orden_galois <= 48:
                try:
                    L = K.galois_closure('b')
                    discL = L.discriminant()
                    galois_root_disc = float(float(abs(discL)) ** (1.0 / orden_galois))
                except Exception:
                    pass

        d_libre = disc.squarefree_part()
        disc_root_field = "Q" if d_libre == 1 else "Q(sqrt(%s))" % d_libre

        aut_orden = len(K.automorphisms())
        num_clases = len(Gfresh.conjugacy_classes())

        try:
            h = int(K.class_number(proof=False))
        except Exception:
            h = None

        entry.update({
            "ok": True,
            "grado": int(n),
            "signatura": [int(sig[0]), int(sig[1])],
            "discriminante": str(disc),
            "discriminante_factorizado": str(disc.factor()),
            "primos_ramificados": [str(p) for p in ramified],
            "root_discriminante": root_disc,
            "galois_root_discriminante": galois_root_disc,
            "galois_root_discriminante_exacto": galois_root_disc_exacto,
            "discriminante_root_field": disc_root_field,
            "grupo_orden": int(orden_galois),
            "grupo_label": str(label),
            "grupo_nombre": nombre,
            "es_galois": es_galois,
            "es_abeliano": es_abeliano,
            "es_ciclico": es_ciclico,
            "aut_orden": int(aut_orden),
            "num_clases_conjugacion": int(num_clases),
            "numero_clase": h,
        })
    except Exception as e:
        entry.update({"ok": False, "error": str(e)})

    resultados.append(entry)

print("===JSON_START===")
print(json.dumps(resultados))
print("===JSON_END===")
'''

In [7]:
def analizar_polinomios(lista_polys: list, var_name: str = 'x', timeout: int = 1800) -> list:
    """Analiza N polinomios en UN solo proceso Sage. Devuelve lista de dicts."""
    polys_json = json.dumps(lista_polys)
    codigo = (_PLANTILLA_SAGE_BATCH
              .replace("%%VAR%%", var_name)
              .replace("%%POLYS_JSON%%", polys_json))

    salida = ejecutar_sage(codigo, timeout=timeout)
    inicio = salida.find("===JSON_START===") + len("===JSON_START===")
    fin = salida.find("===JSON_END===")
    return json.loads(salida[inicio:fin].strip())


def analizar_polinomio(poly_str: str, var_name: str = 'x') -> dict:
    """Analiza UN solo polinomio (reutiliza analizar_polinomios con lista de tamaño 1)."""
    d = analizar_polinomios([poly_str], var_name)[0]
    if not d.get("ok"):
        raise ValueError(f"Error en Sage: {d.get('error')}")
    return d

In [8]:
def _descripcion_grupo(d: dict) -> str:
    orden = d['grupo_orden']
    if d['es_ciclico']:
        return f"A cyclic group of order {orden}"
    if d['es_abeliano']:
        return f"An abelian group of order {orden}"
    return f"A group of order {orden}, isomorphic to {d['grupo_nombre']}"


def _imprimir_reporte(d: dict, poly_str: str):
    print("Invariants")
    print("-" * 60)
    print(f"Polynomial:                {poly_str}")
    print(f"Degree:                    {d['grado']}")
    print(f"Signature:                 ({d['signatura'][0]}, {d['signatura'][1]})")
    print(f"Discriminant:              {d['discriminante']} = {d['discriminante_factorizado']}")
    print(f"Root discriminant:         {d['root_discriminante']:.2f}")

    if d['galois_root_discriminante'] is not None:
        if d['galois_root_discriminante_exacto']:
            print(f"Galois root discriminant:  {d['galois_root_discriminante_exacto']} "
                  f"≈ {d['galois_root_discriminante']:.14f}")
        else:
            print(f"Galois root discriminant:  ≈ {d['galois_root_discriminante']:.14f}")
    else:
        print("Galois root discriminant:  (omitido: grupo demasiado grande)")

    print(f"Ramified primes:           {', '.join(d['primos_ramificados'])}")
    print(f"Discriminant root field:   {d['discriminante_root_field']}")

    if d['es_galois']:
        print(f"Aut(K/Q) = Gal(K/Q):       {d['grupo_nombre']}")
    else:
        print(f"Aut(K/Q):                  order {d['aut_orden']}")
        print(f"Gal(K/Q) [cierre normal]:  {d['grupo_nombre']} ({d['grupo_label']})")

    if d['es_galois'] and d['es_abeliano']:
        print("\nThis field is Galois and abelian over Q.")
    elif d['es_galois']:
        print("\nThis field is Galois over Q, but the Galois group is not abelian.")
    else:
        print("\nThis field is not Galois over Q.")

    print(f"Class number:              {d['numero_clase']}")
    print()
    print("Galois group")
    print("-" * 60)
    print(f"{d['grupo_nombre']} (as {d['grupo_label']}):")
    print()
    print(_descripcion_grupo(d))
    print(f"The {d['num_clases_conjugacion']} conjugacy class representatives for {d['grupo_nombre']}")


def mostrar_reporte(poly_str: str, var_name: str = 'x') -> dict:
    """Modo INDIVIDUAL: analiza e imprime un solo polinomio."""
    d = analizar_polinomio(poly_str, var_name)
    _imprimir_reporte(d, poly_str)
    return d


def mostrar_reportes_multiples(lista_polys: list, var_name: str = 'x') -> list:
    """Modo LOTE: analiza e imprime varios polinomios en un solo proceso Sage."""
    resultados = analizar_polinomios(lista_polys, var_name)
    for d in resultados:
        print("=" * 60)
        if d.get("ok"):
            _imprimir_reporte(d, d["polinomio"])
        else:
            print(f"Polynomial: {d['polinomio']}")
            print(f"ERROR: {d['error']}")
    print("=" * 60)
    return resultados

## Verificacion cruzada: LMFDB (datos reales) + Gemma-2 (informe)

Estas celdas anaden un paso de control de calidad al notebook:

1. Toman el mismo polinomio con Sage.
2. Lo envian al cuadro **"Find"** de LMFDB `https://www.lmfdb.org/NumberField/`,y descargan la ficha JSON oficial del campo.
3. Comparan, campo a campo, los invariantes calculados localmente contra los
   publicados por LMFDB. **Esta comparacion es un chequeo exacto hecho en
   Python** (no se le pide al modelo de lenguaje que decida si dos numeros
   son iguales).
4. Le piden a **Gemma-2** que redacte, en lenguaje natural, un informe corto
   resumiendo el resultado de esa comparacion para un lector.

Importante: Gemma-2 no navega por internet por si solo (es un modelo de
texto, no un agente con navegador). Por eso el acceso a LMFDB se hace con
`requests` en Python, y Gemma-2 solo interviene al final, para redactar el
informe a partir de la tabla de comparacion ya calculada.

usamos un modelo libiano llamado `llama-3.1-8b-instant` a pesar de que la funcion se llame gemma


In [12]:
# Unica dependencia nueva: cliente HTTP (requests ya suele venir instalado)
!pip install -q requests


In [13]:
# Lee la API key de Groq guardada como secreto en Colab
# (icono de la llave -> Add new secret -> nombre: GROQ_API_KEY)
from google.colab import userdata

GROQ_API_KEY = userdata.get("GROQ_API_KEY")


In [19]:
import requests

def llamar_gemma2_api(prompt: str, modelo: str = "llama-3.1-8b-instant", timeout: int = 60) -> str:
    """
    Llama a un LLM alojado en Groq (API compatible con OpenAI /chat/completions).
    No descarga ningun peso: solo manda el prompt por HTTP y recibe el texto.

    Modelos activos en produccion en Groq (julio 2026), cualquiera sirve aqui:
      - "llama-3.1-8b-instant"     (default: rapido y barato)
      - "llama-3.3-70b-versatile"  (mas grande, mas caro)
      - "openai/gpt-oss-20b"       (alternativa recomendada por Groq)
    """
    url = "https://api.groq.com/openai/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {GROQ_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": modelo,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.2,
        "max_tokens": 400,
    }
    r = requests.post(url, headers=headers, json=payload, timeout=timeout)
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"]


In [20]:
import re
import requests


def buscar_lmfdb(poly_str: str, timeout: int = 25) -> dict:
    """
    Usa el mismo cuadro 'Find' de LMFDB (Label, name, polynomial...) para
    localizar la ficha del campo numerico correspondiente al polinomio, y
    luego usa la API oficial y documentada de LMFDB (/api/nf_fields/) para
    bajar el registro completo en JSON.
    """
    headers = {"User-Agent": "Mozilla/5.0 (verificacion-notebook academica)"}
    base = "https://www.lmfdb.org/NumberField/"

    # Paso 1: el "jump" del buscador normaliza el polinomio a su forma
    # canonica y redirige a la ficha del campo (si existe en la base).
    r = requests.get(base, params={"jump": poly_str}, timeout=timeout,
                      headers=headers, allow_redirects=True)
    r.raise_for_status()

    m = re.search(r"/NumberField/(\d+\.\d+\.\d+\.\d+)", r.url)
    if not m:
        return {"encontrado": False, "url_consultada": r.url}
    label = m.group(1)

    # Paso 2: la API publica y documentada de LMFDB (distinta del sitio
    # HTML), pensada para consumo programatico.
    api_url = "https://www.lmfdb.org/api/nf_fields/"
    dr = requests.get(api_url, params={"label": label, "_format": "json"},
                       timeout=timeout, headers=headers)
    dr.raise_for_status()
    payload = dr.json()

    # La API puede envolver los resultados de formas ligeramente distintas
    # segun la version; esto lo maneja sin asumir una unica forma.
    registros = payload.get("data", payload) if isinstance(payload, dict) else payload
    if isinstance(registros, dict):
        registros = [registros]
    if not registros:
        return {
            "encontrado": False,
            "url_consultada": r.url,
            "mensaje": "La API no devolvio registros para ese label.",
        }

    return {
        "encontrado": True,
        "label": label,
        "ficha_url": f"{base}{label}",
        "datos": registros[0],
    }


def _num(x):
    try:
        return float(x)
    except Exception:
        return None


def comparar_automatico(sage_d: dict, lmfdb_d: dict) -> list:
    """
    Comparacion determinista campo a campo (la igualdad numerica la decide
    Python, no el LLM). Devuelve una lista de filas
    (nombre, valor_sage, valor_lmfdb, coincide).
    """
    nf = lmfdb_d["datos"]

    r1_lmfdb = nf["degree"] - 2 * nf["r2"]
    disc_lmfdb = nf["disc_sign"] * int(nf["disc_abs"])
    ramps_lmfdb = sorted(str(p) for p in nf["ramps"])

    filas = []

    def fila(nombre, v_sage, v_lmfdb, coincide):
        filas.append({"campo": nombre, "sage": v_sage, "lmfdb": v_lmfdb, "coincide": coincide})

    fila("grado", sage_d["grado"], nf["degree"], sage_d["grado"] == nf["degree"])
    fila("signatura", tuple(sage_d["signatura"]), (r1_lmfdb, nf["r2"]),
         tuple(sage_d["signatura"]) == (r1_lmfdb, nf["r2"]))
    fila("discriminante", sage_d["discriminante"], str(disc_lmfdb),
         int(sage_d["discriminante"]) == disc_lmfdb)
    fila("primos_ramificados", sorted(sage_d["primos_ramificados"]), ramps_lmfdb,
         sorted(sage_d["primos_ramificados"]) == ramps_lmfdb)
    fila("root_discriminante", round(sage_d["root_discriminante"], 3), round(nf["rd"], 3),
         abs(sage_d["root_discriminante"] - nf["rd"]) < 1e-2)
    fila("grupo_galois_label", sage_d["grupo_label"], nf["galois_label"],
         sage_d["grupo_label"] == nf["galois_label"])
    fila("es_galois", sage_d["es_galois"], nf["is_galois"], sage_d["es_galois"] == nf["is_galois"])
    fila("es_abeliano", sage_d["es_abeliano"], nf["gal_is_abelian"],
         sage_d["es_abeliano"] == nf["gal_is_abelian"])
    fila("es_ciclico", sage_d["es_ciclico"], nf["gal_is_cyclic"],
         sage_d["es_ciclico"] == nf["gal_is_cyclic"])
    numero_clase_lmfdb = _num(nf["class_number"])
    fila("numero_clase", sage_d["numero_clase"], numero_clase_lmfdb,
         sage_d["numero_clase"] == numero_clase_lmfdb)

    return filas


def _prompt_gemma(poly_str: str, label: str, ficha_url: str, filas: list) -> str:
    tabla = "\n".join(
        f"- {f['campo']}: sage={f['sage']!r} | lmfdb={f['lmfdb']!r} | "
        f"{'OK' if f['coincide'] else 'DIFERENCIA'}"
        for f in filas
    )
    return f"""Eres un revisor matematico. Se calcularon invariantes de un campo de
numeros con SageMath para el polinomio:

    {poly_str}

y se compararon automaticamente (comparacion exacta hecha en Python, no por ti)
contra la ficha publica de LMFDB: {label} ({ficha_url}).

Resultado de la comparacion campo a campo:
{tabla}

Escribe un informe breve (maximo 6 lineas) en espanol dirigido a un lector
, indicando si los resultados del notebook coinciden con LMFDB, y si
hay alguna diferencia, cual es y que podria significar (p.ej. redondeo,
convencion de signo, o un error real de calculo). No inventes datos que no
esten en la tabla."""


def verificar_con_gemma(llamar_llm, poly_str: str, sage_d: dict) -> dict:
    """
    Orquesta todo el flujo:
    1) consulta LMFDB para el polinomio (API oficial, no scraping de HTML),
    2) compara deterministicamente contra el resultado local de Sage,
    3) le pide a Gemma-2 (via API) que redacte el informe humano.
    """
    lmfdb_d = buscar_lmfdb(poly_str)
    if not lmfdb_d["encontrado"]:
        return {
            "encontrado_en_lmfdb": False,
            "mensaje": lmfdb_d.get(
                "mensaje",
                "LMFDB no devolvio una ficha de campo numerico para este "
                "polinomio (puede no estar en la base de datos, o el grado/"
                "discriminante excede lo que LMFDB almacena)."
            ),
        }

    filas = comparar_automatico(sage_d, lmfdb_d)
    prompt = _prompt_gemma(poly_str, lmfdb_d["label"], lmfdb_d["ficha_url"], filas)
    informe = llamar_llm(prompt)

    return {
        "encontrado_en_lmfdb": True,
        "label_lmfdb": lmfdb_d["label"],
        "ficha_url": lmfdb_d["ficha_url"],
        "comparacion": filas,
        "todo_coincide": all(f["coincide"] for f in filas),
        "informe_gemma": informe,
    }


In [24]:
def _imprimir_comparacion_lmfdb(poly_str: str, resultado_sage: dict):
    """
    Imprime debajo del reporte de Sage la verificación contra LMFDB
    y el informe generado por el modelo vía API.
    """
    print()
    print("Verificación contra LMFDB")
    print("-" * 60)

    try:
        reporte = verificar_con_gemma(llamar_gemma2_api, poly_str, resultado_sage)
    except Exception as e:
        print("Error verificando contra LMFDB/Groq:", e)
        return None

    print(f"Encontrado en LMFDB: {reporte['encontrado_en_lmfdb']}")

    if reporte["encontrado_en_lmfdb"]:
        print(f"Ficha LMFDB: {reporte['ficha_url']}")
        print(f"Todo coincide: {reporte['todo_coincide']}")
        print()

        print("Comparación Sage vs LMFDB")
        print("-" * 60)
        for f in reporte["comparacion"]:
            estado = "OK" if f["coincide"] else "DIFERENCIA"
            print(f"  [{estado}] {f['campo']}: sage={f['sage']}  lmfdb={f['lmfdb']}")

        print()
        print("Informe de Gemma-2:")
        print(reporte["informe_gemma"])
    else:
        print(reporte["mensaje"])

    return reporte


def consultar():
    """
    Menú principal unificado:
    1) muestra el reporte de Sage como antes;
    2) debajo muestra la verificación contra LMFDB;
    3) debajo muestra la comparación y el informe generado por la API.
    """
    print("¿Cuántos polinomios quieres analizar?")
    print("  1) Uno solo")
    print("  2) Varios (lote)")
    modo = input("Elige 1 o 2: ").strip()

    if modo == "1":
        entrada = input("Ingresa el polinomio en variable x (ej: x^5 - 2): ").strip()

        if not entrada:
            print("No ingresaste nada.")
            return

        try:
            # Primero imprime el resultado de Sage igual que antes
            resultado_sage = mostrar_reporte(entrada)

            # Después imprime la verificación y comparación
            _imprimir_comparacion_lmfdb(entrada, resultado_sage)

        except Exception as e:
            print("Error:", e)

    elif modo == "2":
        print("Ingresa los polinomios separados por ';' (ej: x^5 - 2; x^3 - 2; x^4 + 1)")
        entrada = input("> ").strip()

        if not entrada:
            print("No ingresaste nada.")
            return

        lista = [p.strip() for p in entrada.split(";") if p.strip()]

        try:
            # Analiza todos los polinomios con Sage en un solo proceso
            resultados_sage = analizar_polinomios(lista)

            for resultado_sage in resultados_sage:
                print("=" * 60)

                if resultado_sage.get("ok"):
                    poly_actual = resultado_sage["polinomio"]

                    # Primero imprime el reporte de Sage
                    _imprimir_reporte(resultado_sage, poly_actual)

                    # Después imprime la verificación y comparación
                    _imprimir_comparacion_lmfdb(poly_actual, resultado_sage)
                else:
                    print(f"Polynomial: {resultado_sage['polinomio']}")
                    print(f"ERROR en Sage: {resultado_sage['error']}")
                    print("No se consulta LMFDB porque Sage no pudo analizar este polinomio.")

            print("=" * 60)

        except Exception as e:
            print("Error:", e)

    else:
        print("Opción no válida. Elige 1 o 2.")


consultar()


¿Cuántos polinomios quieres analizar?
  1) Uno solo
  2) Varios (lote)
Elige 1 o 2: 2
Ingresa los polinomios separados por ';' (ej: x^5 - 2; x^3 - 2; x^4 + 1)
> x^5 - 2; x^3 - 2; x^4 + 1
Invariants
------------------------------------------------------------
Polynomial:                x^5 - 2
Degree:                    5
Signature:                 (1, 2)
Discriminant:              50000 = 2^4 * 5^5
Root discriminant:         8.71
Galois root discriminant:  ≈ 11.08254495193135
Ramified primes:           2, 5
Discriminant root field:   Q(sqrt(5))
Aut(K/Q):                  order 1
Gal(K/Q) [cierre normal]:  C5 : C4 (5T3)

This field is not Galois over Q.
Class number:              1

Galois group
------------------------------------------------------------
C5 : C4 (as 5T3):

A group of order 20, isomorphic to C5 : C4
The 5 conjugacy class representatives for C5 : C4

Verificación contra LMFDB
------------------------------------------------------------
Encontrado en LMFDB: True
Ficha LMF

BLOQUES DE PRUEBA